Na etapa de EDA, o grupo deve realizar uma análise exploratória abrangente para entender a estrutura, qualidade, distribuições, relações entre variáveis (numéricas e categóricas) e desafios, como missing values e vieses, preparando para encoding e modelagem nas fases subsequentes.
Descrição do Dataset: O dataset Adult consiste em 48.842 instâncias, cada uma representando um indivíduo com 14 features mistas e o target 'income'. Valores ausentes são representados por '?'. Use o conjunto completo para EDA, mas amostras estratificadas (por income) para visualizações eficientes se necessário.

Tarefas básicas de EDA:

1. Carregamento e inspeção inicial dos dados:

    - Explicar cada feature do dataset;
    - Verificar o número de instâncias e features.
    - Identificar tipos de dados (numéricos, categóricos).
    - Detectar valores ausentes ('?') e inconsistências.
    - Analisar a distribuição do target 'income'.

In [12]:
import pandas as pd
import numpy as np

# Ler arquivo de documentação para entender o dataset
with open('adult/adult.names', 'r') as f:
    doc_lines = f.readlines()

print("\nNomes das features encontrados na documentação:\n")
feature_lines = []
for line in doc_lines[96:200]:  # Linhas onde estão os nomes das features
    if ':' in line and not line.startswith('|'):
        feature_name = line.split(':')[0].strip()
        feature_lines.append(line.strip())
        print(f"  {line.strip()}")

# Definir nomes das colunas baseado na documentação
column_names = [line.split(':')[0].strip() for line in feature_lines]
column_names.append('income')

print("\nColunas finais usadas no DataFrame:")
print(column_names)

# Carregar dataset
df = pd.read_csv('adult/adult.data', names=column_names, skipinitialspace=True)


Nomes das features encontrados na documentação:

  age: continuous.
  workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
  fnlwgt: continuous.
  education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
  education-num: continuous.
  marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
  occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
  relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
  race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
  sex: Female, Male.
  capital-gain: continuous.
  capital-loss: continuous.
  hours-per-w

## 1. Explicação de cada Feature do Dataset

In [11]:
print("1. EXPLICAÇÃO DE CADA FEATURE")

# Analisar tipos de dados diretamente do DataFrame
for col, dtype in df.dtypes.items():
    print(f"  {col:20s}: {dtype}")

# Descrições baseadas na documentação + análise dos dados
feature_descriptions = {
    'age': 'Idade do indivíduo',
    'workclass': 'Tipo de empregador',
    'fnlwgt': 'Peso amostral - representa quantas pessoas o registro representa na população',
    'education': 'Nível educacional',
    'education-num': 'Nível educacional convertido em número',
    'marital-status': 'Estado civil',
    'occupation': 'Tipo de ocupação/profissão',
    'relationship': 'Papel na família',
    'race': 'Raça/etnia',
    'sex': 'Sexo',
    'capital-gain': 'Ganhos de capital financeiro',
    'capital-loss': 'Perdas de capital financeiro',
    'hours-per-week': 'Horas trabalhadas por semana',
    'native-country': 'País de origem',
    'income': 'TARGET - Se ganha mais que $50K/ano'
}

for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    desc = feature_descriptions.get(col, 'Sem descrição')
    
    # Determinar tipo baseado no dtype do pandas
    if dtype in ['int64', 'float64']:
        tipo = 'NUMÉRICA'
    else:
        tipo = 'CATEGÓRICA'
    
    print(f"\n{i:2d}. {col:20s}")
    print(f"    Descrição: {desc}")
    print(f"    Tipo: {tipo} (dtype: {dtype})")

1. EXPLICAÇÃO DE CADA FEATURE
  age                 : int64
  workclass           : object
  fnlwgt              : int64
  education           : object
  education-num       : int64
  marital-status      : object
  occupation          : object
  relationship        : object
  race                : object
  sex                 : object
  capital-gain        : int64
  capital-loss        : int64
  hours-per-week      : int64
  native-country      : object
  income              : object

 1. age                 
    Descrição: Idade do indivíduo
    Tipo: NUMÉRICA (dtype: int64)

 2. workclass           
    Descrição: Tipo de empregador
    Tipo: CATEGÓRICA (dtype: object)

 3. fnlwgt              
    Descrição: Peso amostral - representa quantas pessoas o registro representa na população
    Tipo: NUMÉRICA (dtype: int64)

 4. education           
    Descrição: Nível educacional
    Tipo: CATEGÓRICA (dtype: object)

 5. education-num       
    Descrição: Nível educacional convertido e

## 2. Verificar Número de Instâncias e Features

In [14]:
print("2. NÚMERO DE INSTÂNCIAS E FEATURES")

print(f"\nNúmero de instâncias (linhas): {df.shape[0]:,}")
print(f"Número de features (colunas): {df.shape[1]}")
print(f"  - Features preditoras: {df.shape[1] - 1}")
print(f"  - Target (income): 1")

print("\nPrimeiras 3 linhas do dataset:")
print(df.head(3))

2. NÚMERO DE INSTÂNCIAS E FEATURES

Número de instâncias (linhas): 32,561
Número de features (colunas): 15
  - Features preditoras: 14
  - Target (income): 1

Primeiras 3 linhas do dataset:
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   

       marital-status         occupation   relationship   race   sex  \
0       Never-married       Adm-clerical  Not-in-family  White  Male   
1  Married-civ-spouse    Exec-managerial        Husband  White  Male   
2            Divorced  Handlers-cleaners  Not-in-family  White  Male   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0              13  United-States  <=50K  
2             0             0              40  United-States  

## 3. Identificar Tipos de Dados (Numéricos e Categóricos)

In [19]:
print("3. IDENTIFICAÇÃO DE TIPOS DE DADOS")

# Separar por tipo
numerical = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical = df.select_dtypes(include=['object']).columns.tolist()

# Remover target das listas
if 'income' in categorical:
    categorical.remove('income')

print(f"\nFeatures NUMÉRICAS ({len(numerical)}):")
for feat in numerical:
    print(f"  - {feat}")

print(f"\nFeatures CATEGÓRICAS ({len(categorical)}):")
for feat in categorical:
    print(f"  - {feat}")

3. IDENTIFICAÇÃO DE TIPOS DE DADOS

Features NUMÉRICAS (6):
  - age
  - fnlwgt
  - education-num
  - capital-gain
  - capital-loss
  - hours-per-week

Features CATEGÓRICAS (8):
  - workclass
  - education
  - marital-status
  - occupation
  - relationship
  - race
  - sex
  - native-country


## 4. Detectar Valores Ausentes ('?') e Inconsistências

In [21]:
print("4. DETECÇÃO DE VALORES AUSENTES E INCONSISTÊNCIAS")

# Detectar '?' nas features categóricas
for col in categorical + ['income']:
    count = (df[col] == '?').sum()
    if count > 0:
        pct = (count / len(df)) * 100
        print(f"  - {col:20s}: {count:5,} ({pct:.2f}%)")

# Total de linhas com missing
linhas_com_missing = df[df.isin(['?']).any(axis=1)].shape[0]
print(f"\nTotal de instâncias com valores ausentes: {linhas_com_missing:,} ({linhas_com_missing/len(df)*100:.2f}%)")

# Verificar duplicatas
duplicatas = df.duplicated().sum()
print(f"Linhas duplicadas: {duplicatas:,}")

4. DETECÇÃO DE VALORES AUSENTES E INCONSISTÊNCIAS
  - workclass           : 1,836 (5.64%)
  - occupation          : 1,843 (5.66%)
  - native-country      :   583 (1.79%)

Total de instâncias com valores ausentes: 2,399 (7.37%)
Linhas duplicadas: 24


## 5. Analisar a Distribuição do Target 'income'

In [24]:
print("5. DISTRIBUIÇÃO DO TARGET 'INCOME'")

# Contar classes
distribuicao = df['income'].value_counts()
percentual = df['income'].value_counts(normalize=True) * 100

# Distribuição das classes
for classe, qtd in distribuicao.items():
    pct = percentual[classe]
    print(f"  {classe:8s}: {qtd:6,} ({pct:5.2f}%)")

print(f"  {'TOTAL':8s}: {len(df):6,} (100.00%)")

# Análise de balanceamento
razao = distribuicao.max() / distribuicao.min()
print(f"\nRazão entre classe majoritária/minoritária: {razao:.2f}:1")

if razao > 2:
    print(f"\nCONCLUSÃO: Dataset DESBALANCEADO")
else:
    print(f"\nCONCLUSÃO: Dataset balanceado")

5. DISTRIBUIÇÃO DO TARGET 'INCOME'
  <=50K   : 24,720 (75.92%)
  >50K    :  7,841 (24.08%)
  TOTAL   : 32,561 (100.00%)

Razão entre classe majoritária/minoritária: 3.15:1

CONCLUSÃO: Dataset DESBALANCEADO
